# Section 3. 장바구니 분석 — 최종본

## 0. 공통 설정

In [ ]:
import pandas as pd
import numpy as np
import gc
import glob
import os
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

TRAIN_CSV_PATH = 'cosmetics_train.csv'


---
## 3-1. 담긴 상품의 최종 운명

In [ ]:
train = pd.read_csv(TRAIN_CSV_PATH, usecols=["user_hash", "product_id", "outcome"])

print(f"전체 행(카트 라인) 수: {len(train):,}")
n_pairs = train.groupby(["user_hash", "product_id"]).ngroups
print(f"유니크 (유저,상품) 쌍 수: {n_pairs:,}")
print(f"유니크 유저 수: {train['user_hash'].nunique():,}")
print(f"유니크 상품 수: {train['product_id'].nunique():,}")

# 벡터화: outcome을 boolean 컬럼으로 먼저 만든 뒤 groupby().any() (커스텀 apply보다 훨씬 빠름)
train["is_purchased"] = train["outcome"] == "purchased"
train["is_removed"] = train["outcome"] == "explicitly_removed"

grp = train.groupby(["user_hash", "product_id"], sort=False).agg(
    has_purchase=("is_purchased", "any"),
    has_removed=("is_removed", "any"),
)

grp["final"] = np.select(
    [grp["has_purchase"], grp["has_removed"]],
    ["구매됨", "삭제후미구매"],
    default="방치됨",
)

pair_count = grp["final"].value_counts().reindex(["방치됨", "삭제후미구매", "구매됨"])
pair_pct = (pair_count / pair_count.sum() * 100).round(2)
result_3_1 = pd.DataFrame({"건수": pair_count, "비율(%)": pair_pct})
print(result_3_1)

del train, grp
gc.collect()


---
## 3-2 & 3-3. 원본 이벤트 로그 기반 (REES46 cosmetics shop)

In [ ]:
%pip install -q kagglehub

import kagglehub
dataset_path = kagglehub.dataset_download("mkechinov/ecommerce-events-history-in-cosmetics-shop")
print("Dataset path:", dataset_path)

csv_files = sorted(glob.glob(os.path.join(dataset_path, "*.csv")))
print(csv_files)

usecols = ["event_time", "event_type", "product_id", "user_id", "user_session"]
dfs = []
for f in csv_files:
    tmp = pd.read_csv(f, usecols=usecols, dtype={"event_type": "category"})
    dfs.append(tmp)
df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()
print("원본 행 수:", len(df))

df = df.dropna(subset=["user_session"]).copy()
df["event_time"] = pd.to_datetime(df["event_time"], utc=True).dt.tz_localize(None)
df["user_session"] = df["user_session"].astype("category")
gc.collect()
print("정제 후 행 수:", len(df))


### 3-2. 구매 시점 분석


In [ ]:
cart_first = df[df["event_type"]=="cart"].groupby(["user_id","product_id"])["event_time"].min().rename("cart_time")
cart_first_session = (df[df["event_type"]=="cart"].sort_values("event_time")
                       .groupby(["user_id","product_id"]).first()[["user_session"]]
                       .rename(columns={"user_session":"cart_session"}))

purchase_events = df[df["event_type"]=="purchase"][["user_id","product_id","event_time","user_session"]].copy()
cart_purchase = purchase_events.merge(cart_first, on=["user_id","product_id"], how="inner")
cart_purchase = cart_purchase[cart_purchase["event_time"] >= cart_purchase["cart_time"]]

# 같은 (유저,상품)에 구매가 여러 건이면, 카트 이후 "첫" 구매만 사용
cart_purchase = cart_purchase.sort_values("event_time").groupby(["user_id","product_id"]).first().reset_index()
cart_purchase = cart_purchase.merge(cart_first_session, on=["user_id","product_id"], how="left")

cart_purchase["delay_min"] = (cart_purchase["event_time"] - cart_purchase["cart_time"]).dt.total_seconds() / 60
cart_purchase["same_session"] = cart_purchase["user_session"].astype(str) == cart_purchase["cart_session"].astype(str)

same_session_pct = cart_purchase["same_session"].mean() * 100
diff_session_pct = 100 - same_session_pct

print(f"cart & purchase 모두 있는 (유저,상품) 쌍: {len(cart_purchase):,}개")
print(f"같은 세션에서 구매: {same_session_pct:.1f}%")
print(f"다른 세션에서 구매: {diff_session_pct:.1f}%")

bins = [-0.01, 60, 60*24, 60*24*7, np.inf]
labels = ["1시간 이내", "1-24시간", "1-7일", "7일 초과"]
cart_purchase["delay_bucket"] = pd.cut(cart_purchase["delay_min"], bins=bins, labels=labels)
delay_dist = cart_purchase["delay_bucket"].value_counts(normalize=True).reindex(labels) * 100
print("\n지연시간 분포:")
print(delay_dist)


In [ ]:
def mode_group(b):
    if b == "1시간 이내":
        return "즉시"
    elif b == "1-24시간":
        return "당일"
    else:
        return "지연"

cart_purchase["purchase_mode"] = cart_purchase["delay_bucket"].apply(mode_group)

user_mode = cart_purchase.groupby("user_id")["purchase_mode"].agg(lambda x: x.mode().iloc[0])
user_purchase_count = df[df["event_type"]=="purchase"].groupby("user_id").size().rename("purchase_count")

user_summary = pd.concat([user_mode, user_purchase_count], axis=1).dropna()
mode_summary = user_summary.groupby("purchase_mode").agg(
    유저수=("purchase_count", "count"),
    평균구매건수=("purchase_count", "mean"),
).reindex(["즉시", "당일", "지연"])
print(mode_summary)

del cart_first, cart_first_session, purchase_events, cart_purchase, user_mode, user_purchase_count, user_summary
gc.collect()


### 3-3. 세션 유형 비교


In [ ]:
sess_pivot = df.pivot_table(index="user_session", columns="event_type",
                             values="user_id", aggfunc="size", fill_value=0, observed=True)
for col in ["view", "cart", "purchase"]:
    if col not in sess_pivot.columns:
        sess_pivot[col] = 0

sessions = pd.DataFrame({
    "n_view": sess_pivot["view"],
    "n_cart": sess_pivot["cart"],
    "n_purchase": sess_pivot["purchase"],
}).reset_index()
sessions["has_view"] = sessions["n_view"] > 0
sessions["has_purchase"] = sessions["n_purchase"] > 0
del sess_pivot
gc.collect()

session_type_conv = sessions.groupby("has_view")["has_purchase"].mean() * 100
session_type_count = sessions.groupby("has_view").size()
session_type_share = (session_type_count / session_type_count.sum() * 100).round(1)

result_3_3a = pd.DataFrame({
    "세션수": session_type_count,
    "비중%": session_type_share,
    "전환율%": session_type_conv.round(2),
})
print(result_3_3a)

ratio = session_type_conv[False] / session_type_conv[True]
print(f"\n조회 0회 세션이 조회 1회+ 세션보다 전환율이 {ratio:.2f}배")


In [ ]:
purchase_sessions = sessions[sessions["has_purchase"]]

pct_zero_view = (purchase_sessions["n_view"] == 0).mean() * 100
pct_zero_cart = (purchase_sessions["n_cart"] == 0).mean() * 100

print(f"구매 세션({len(purchase_sessions):,}개) 중 '조회 0회' 비율: {pct_zero_view:.1f}%")
print(f"구매 세션 중 '담기 0회'(이전 세션 카트를 결제만 함) 비율: {pct_zero_cart:.1f}%")

result_3_3b = pd.DataFrame({
    "구매세션": [len(purchase_sessions), purchase_sessions["n_view"].mean(), purchase_sessions["n_cart"].mean()],
    "비구매세션": [len(sessions)-len(purchase_sessions),
               sessions.loc[~sessions["has_purchase"], "n_view"].mean(),
               sessions.loc[~sessions["has_purchase"], "n_cart"].mean()],
}, index=["세션수", "평균조회", "평균담기"])
print(result_3_3b)
